In [1]:

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [2]:
import os
from tensorflow.keras.preprocessing.image import  ImageDataGenerator
from keras.layers import Dense,Flatten,GlobalAveragePooling2D,Input,Conv2D,MaxPooling2D,Dropout
from keras.models import Model,load_model,Sequential
from keras.applications.xception import  Xception
from keras.applications.vgg16 import VGG16
from keras.applications.vgg19 import VGG19
from keras.applications.resnet50 import  ResNet50
from keras.applications.inception_v3 import InceptionV3
from keras.applications.inception_resnet_v2 import InceptionResNetV2
from keras.applications.mobilenet import MobileNet
import keras.callbacks as kcallbacks
import keras
from keras.preprocessing.image import load_img,img_to_array
import math
import random
from keras.utils import plot_model
import matplotlib.pyplot as plt
import sklearn.metrics as metrics
import numpy as np
from PIL import Image
import sklearn.metrics as metrics
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

2026-09-29 22:39:49.883963: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TARGET_SIZE = (224, 224)
INPUT_SIZE = (224, 224, 3)
BATCHSIZE = 128

# 1. Normalization (Remove validation_split here)
# It is best practice to use separate generator instances, 
# even if both just rescale, to prevent future accidental leakage if you add augmentation later.
train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Update these classes based on your actual CICIDS2017 labels
wanted_classes = ['DoS', 'Fuzzy', 'RPM', 'Advanced', 'Patator', 'BENIGN']

# 2. Train Generator (Points directly to the 'train' folder)
train_generator = train_datagen.flow_from_directory(
    directory='../new_Images/train/', # Update this to your actual train images path
    classes=wanted_classes, 
    target_size=TARGET_SIZE,
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=True, # Shuffle training data
    seed=42
)

# 3. Validation Generator (Points directly to the 'val' folder)
validation_generator = val_test_datagen.flow_from_directory(
    directory='../new_Images/val/', # Update this to your actual val images path
    classes=wanted_classes, 
    target_size=TARGET_SIZE,
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=False, # Do not shuffle validation data
    seed=42
)


Found 1710819 images belonging to 6 classes.
Found 565575 images belonging to 6 classes.


In [5]:
class LossHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = {'batch':[], 'epoch':[]}
        self.accuracy = {'batch':[], 'epoch':[]}
        self.val_loss = {'batch':[], 'epoch':[]}
        self.val_accuracy = {'batch':[], 'epoch':[]}
    def on_batch_end(self, batch, logs={}):
        self.losses['batch'].append(logs.get('loss'))
        self.accuracy['batch'].append(logs.get('acc'))
        self.val_loss['batch'].append(logs.get('val_loss'))
        self.val_accuracy['batch'].append(logs.get('val_accuracy'))
    def on_epoch_end(self, batch, logs={}):
        self.losses['epoch'].append(logs.get('loss'))
        self.accuracy['epoch'].append(logs.get('acc'))
        self.val_loss['epoch'].append(logs.get('val_loss'))
        self.val_accuracy['epoch'].append(logs.get('val_accuracy'))
    def loss_plot(self, loss_type):
        iters = range(len(self.losses[loss_type]))
        plt.figure()
        plt.plot(iters, self.losses[loss_type], 'g', label='train loss')
        if loss_type == 'epoch':
            # acc
            plt.plot(iters, self.accuracy[loss_type], 'r', label='train acc')
            # loss
            plt.plot(iters, self.losses[loss_type], 'g', label='train loss')
            # val_acc
            plt.plot(iters, self.val_accuracy[loss_type], 'b', label='val_accuracy')
            # val_loss
            plt.plot(iters, self.val_loss[loss_type], 'k', label='val loss')
        plt.grid(True)
        plt.xlabel(loss_type)
        plt.ylabel('acc-loss')
        plt.legend(loc="upper right")
        plt.show()
history_this= LossHistory()

In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
import numpy as np
from tensorflow import keras
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import callbacks as kcallbacks
from sklearn.metrics import classification_report

# Import the new models
from tensorflow.keras.applications import ResNet50V2, MobileNetV2, DenseNet121

def train_cnn_model(base_model, model_name, num_class, epochs, train_gen, val_gen, history_obj, input_shape=(224,224,3)):
    """
    Dynamically builds, fine-tunes, and trains a given CNN base model.
    """
    # 1. Dynamic Layer Freezing (Freeze the first 75% of the network)
    total_layers = len(base_model.layers)
    freeze_cutoff = int(total_layers * 0.5)
    
    for layer in base_model.layers[:freeze_cutoff]:
        layer.trainable = False
    for layer in base_model.layers[freeze_cutoff:]:
        layer.trainable = True

    # 2. Build the Custom Top Layers
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(units=256, activation='relu')(x)
    x = Dropout(0.6)(x)
    predictions = Dense(num_class, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions, name=model_name)
    
    # 3. Compile Model
    opt = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.8, beta_2=0.999, epsilon=1e-08)
    model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])
    
    # 4. Callbacks
    savepath = f"{model_name}_best.keras"
    earlyStopping = kcallbacks.EarlyStopping(
        monitor='val_accuracy', patience=2, verbose=1, mode='max')
    saveBestModel = kcallbacks.ModelCheckpoint(
        filepath=savepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
    
    # 5. Train Model
    print(f"--- Starting Training for {model_name} ---")
    hist = model.fit(
        train_gen,
        steps_per_epoch=len(train_gen),
        epochs=epochs,
        validation_data=val_gen,
        validation_steps=len(val_gen),
        callbacks=[earlyStopping, saveBestModel, history_obj]
    )
    
    # 6. Evaluation and Classification Report
    print(f"--- Classification Report for {model_name} ---")
    y_true = val_gen.classes  
    y_pred = model.predict(val_gen)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    class_names = list(val_gen.class_indices.keys())
    print(classification_report(y_true, y_pred_classes, target_names=class_names))
    
    return model, hist

In [7]:
NUM_CLASSES = 6
EPOCHS = 5
INPUT_SHAPE = (224, 224, 3)

# --- 1. ResNet50V2 ---
resnet_base = ResNet50V2(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)
resnet_model, resnet_history = train_cnn_model(
    base_model=resnet_base, 
    model_name='ResNet50V2', 
    num_class=NUM_CLASSES, 
    epochs=EPOCHS, 
    train_gen=train_generator, 
    val_gen=validation_generator, 
    history_obj=history_this, # Assuming history_this is a custom callback you defined earlier
    input_shape=INPUT_SHAPE
)

I0000 00:00:1790714483.631263     495 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


--- Starting Training for ResNet50V2 ---


/home/minyar/tf-venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


2026-09-29 22:41:39.752205: I external/local_xla/xla/service/service.cc:163] XLA service 0x709e78002da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-09-29 22:41:39.752238: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-09-29 22:41:40.085788: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-09-29 22:41:42.318301: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90501
2026-09-29 22:41:43.335071: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10573', 4 bytes spill stores, 4 bytes spill loads

2026-09-29 22:41:44.181857: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:3

 3445/13366 ━━━━━━━━━━━━━━━━━━━━ 1:00:58 369ms/step - accuracy: 0.9673 - loss: 0.0905

2026-09-29 23:03:24.624697: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10283', 8 bytes spill stores, 8 bytes spill loads

2026-09-29 23:03:25.226945: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10573', 4 bytes spill stores, 4 bytes spill loads

2026-09-29 23:03:25.491396: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10575', 112 bytes spill stores, 112 bytes spill loads

2026-09-29 23:03:24.907224: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10232', 10232 bytes spill stores, 10436 bytes spill loads



13366/13366 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.9755 - loss: 0.0636

2026-09-30 00:19:45.040876: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 10232 bytes spill stores, 10436 bytes spill loads




Epoch 1: val_accuracy improved from -inf to 0.98236, saving model to ResNet50V2_best.keras
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 5910s 439ms/step - accuracy: 0.9755 - loss: 0.0636 - val_accuracy: 0.9824 - val_loss: 0.0377
Epoch 2/5
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.9824 - loss: 0.0402
Epoch 2: val_accuracy improved from 0.98236 to 0.98327, saving model to ResNet50V2_best.keras
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 5833s 436ms/step - accuracy: 0.9824 - loss: 0.0402 - val_accuracy: 0.9833 - val_loss: 0.0373
Epoch 3/5
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.9828 - loss: 0.0384
Epoch 3: val_accuracy improved from 0.98327 to 0.98398, saving model to ResNet50V2_best.keras
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 5812s 435ms/step - accuracy: 0.9828 - loss: 0.0384 - val_accuracy: 0.9840 - val_loss: 0.0371
Epoch 4/5
13366/13366 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.9831 - loss: 0.0377
Epoch 4: val_accuracy improved from 0.98398 to 0.98412, saving model 